# AI Literacy Lab: Model Training and Fine-tuning

This notebook contains the coding exercises for the lab.  
Each task has its own section below - **only work on the task your instructor tells you to.** Do not move on to the next task until the class is told they can move on.

| Section | What it covers |
|---------|---------------|
| **Task 2** | Loading and testing GPT-2 |
| **Task 3** | Experimenting with different sampling methods |
| **Task 4** | Fine-tuning GPT-2 on new data |

---
## Task 2: Loading GPT-2

In this section we load GPT-2, a publicly available language model with 124 million parameters.  
Run the cells below **in order**. The first cell may take a few minutes as it downloads the model.

---
### Before You Start

Make sure you have watched the short video on the lab site that explains how to open this notebook in Google Colab and how to enable the **free GPU**. You will need the GPU to run the code efficiently.

**How to run a cell:** Click the **play button** (▶) to the left of any code cell, or press **Cmd + Enter** on Mac, or **Ctrl + Enter** on Windows/Linux.


### Setup

The cell below installs the required Python libraries. **If you are using Google Colab, you can skip this cell** as these libraries are already pre-installed. If you are running this notebook elsewhere, run it once before proceeding.


In [ ]:
# Only run this if you are NOT using Google Colab
!pip install -q transformers datasets accelerate

Run the cell below to import the libraries and set up the environment. This will take a few minutes.


In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("Tip: Go to Runtime > Change runtime type > GPU to speed things up!")

# Load the GPT-2 model and tokenizer
print("Loading GPT-2 model... this may take a few minutes.")

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

# GPT-2 needs a pad token setting
tokenizer.pad_token = tokenizer.eos_token

print("GPT-2 loaded successfully!")
print(f"Model size: {sum(p.numel() for p in model.parameters()):,} parameters")

# This helper function generates text from a prompt.
# You do NOT need to modify this cell - just run it once.

def generate_text(prompt, max_length=100, **kwargs):
    """Generate text from a prompt using the current model."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=max_length,
            pad_token_id=tokenizer.eos_token_id,
            **kwargs
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

print("Generation function ready!")

In [ ]:
# Let's test GPT-2 with a simple prompt!
# Try changing the text in quotes to see different outputs

prompt = "Once upon a time"

output = generate_text(prompt, max_length=80)
print(output)

Done with task 2? Help the people around you. Please wait to move on to task 3 until the teacher directs you.

---
## Task 3: Token Prediction - Sampling Methods

When GPT-2 predicts the next token, it produces a list of probabilities for every possible word.  
But how does it **choose** which one to actually output? That depends on the **sampling method**.

The cell below runs the **same prompt** through four different sampling methods so you can compare the outputs side by side. Each method uses a different strategy to pick the next token from the probability list. The samples we will try are:

- **Greedy**
- **5-Beam Search**
- **Contrastive Search**
- **Random Sampling**

**What to change:**
- Edit the `prompt` (the text in quotes) to try different inputs
- Edit `max_length` to generate more or fewer tokens
- Run the cell and compare how the four methods differ

As a group, work on **group discussion question 1**: 

> As a group, create a table with five columns: Prompt, Greedy Output, 5-Beam Output, Contrastive Search Output, and Random Sampler Output. Try at least 5 different prompts, fill in the table, and compare the results.



In [ ]:
# Change the prompt and max_length below, then run this cell.
# All four methods will use the same prompt so you can compare them.

prompt = "I like playing Minecraft"     # <-- change this
max_length = 100                        # <-- change this

# --- Method 1: Greedy ---
# Picks the single most likely token at every step.
output = generate_text(prompt, max_length, do_sample=False)
print("--- Greedy Output ---")
print(output)

print()

# --- Method 2: 5-Beam Search ---
# Explores multiple sequences in parallel and picks the best overall.
output = generate_text(prompt, max_length, num_beams=5, no_repeat_ngram_size=2, early_stopping=True)
print("--- 5-Beam Output ---")
print(output)

print()

# --- Method 3: Contrastive Search ---
# Balances likelihood with diversity to reduce repetition.
output = generate_text(prompt, max_length, penalty_alpha=0.6, top_k=4, custom_generate="transformers-community/contrastive-search", trust_remote_code=True)
print("--- Contrastive Search Output ---")
print(output)

print()

# --- Method 4: Random Sampling ---
# Randomly picks from the top likely tokens, adding variety.
output = generate_text(prompt, max_length, do_sample=True, top_k=50, temperature=0.7)
print("--- Random Sampling Output ---")
print(output)


### Done with Task 3?

Head back to the tutorial website to complete **discussion questions 2 and 3** with your group.


---
## Task 4: Fine-Tuning GPT-2

In this section we **fine-tune** GPT-2 - continuing its training on a small, focused dataset so that its outputs shift toward a particular style or topic.

We start below with some examples.

### Example 1: Fine-tune on a cooking recipe dataset

The code below fine-tunes GPT-2 on a small collection of cooking-related sentences. After fine-tuning, the model should be more likely to generate recipe-style text.

**Just run the cells below in order** - you do not need to modify any of the code.

In [ ]:
# Import the libraries we need for fine-tuning
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

# This is our training data - a list of cooking-related sentences.
# The model will learn from the patterns in these sentences.
recipe_texts = [
    "Preheat the oven to 180 degrees Celsius and line a baking tray with parchment paper.",
    "Mix the flour, sugar, and baking powder together in a large bowl until well combined.",
    "Melt the butter in a saucepan over low heat, then stir in the chocolate chips until smooth.",
    "Dice the onions and garlic finely, then sauté in olive oil until translucent.",
    "Season the chicken with salt, pepper, and paprika before placing it in the oven.",
    "Bring a large pot of salted water to a boil and cook the pasta for 8 to 10 minutes.",
    "Whisk the eggs, milk, and vanilla extract together until the mixture is light and fluffy.",
    "Spread the tomato sauce evenly over the pizza dough, then add your favourite toppings.",
    "Simmer the soup on low heat for 30 minutes, stirring occasionally to prevent sticking.",
    "Fold the whipped cream gently into the chocolate mixture to keep the batter airy.",
    "Grill the vegetables on high heat for 5 minutes on each side until slightly charred.",
    "Roll out the pastry dough on a floured surface until it is about 3 millimetres thick.",
    "Toast the bread slices until golden brown, then spread avocado and a pinch of chilli flakes.",
    "Marinate the salmon in soy sauce, ginger, and honey for at least 20 minutes before cooking.",
    "Combine the rice, coconut milk, and sugar in a pot and cook on medium heat until thickened.",
    "Chop the fresh herbs including basil, parsley, and coriander to sprinkle over the finished dish.",
    "Beat the butter and sugar together until pale and creamy before adding the eggs one at a time.",
    "Layer the lasagne sheets with bolognese sauce and bechamel, finishing with a layer of cheese.",
    "Knead the bread dough for 10 minutes until it becomes smooth and elastic to the touch.",
    "Roast the sweet potatoes at 200 degrees for 25 minutes until soft and caramelised on the edges.",
    "Stir the risotto continuously while adding warm stock one ladle at a time.",
    "Blend the bananas, strawberries, yoghurt, and ice together to make a smooth fruit smoothie.",
    "Coat the fish fillets in seasoned flour, then pan-fry in butter until crispy and golden.",
    "Let the cake cool completely in the tin before turning it out and adding the icing.",
    "Toss the salad leaves with olive oil, lemon juice, salt, and freshly ground black pepper.",
]

# Convert our list of sentences into a Dataset object that the trainer can use
dataset = Dataset.from_dict({"text": recipe_texts})

# Tokenise the text - this converts each sentence into numbers (token IDs)
# that the model can understand, just like we saw in Task 1
def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])
print(f"Dataset prepared: {len(recipe_texts)} recipe examples")

In [ ]:
# Set up the training configuration
training_args = TrainingArguments(
    output_dir="./fine_tuned_recipes",  # Where to save the model
    num_train_epochs=5,                  # How many times to loop through the data
    per_device_train_batch_size=2,       # How many examples to process at once
    learning_rate=5e-5,                  # How much to adjust the model each step
    save_strategy="no",                  # Don't save checkpoints (saves disk space)
    logging_steps=5,                     # Print progress every 5 steps
    report_to="none",                    # Don't send logs to external services
)

# The data collator handles formatting the data for language modelling
# mlm=False means we are doing causal (next-token) language modelling, not masked
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# The Trainer brings everything together: the model, the data, and the settings
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Start fine-tuning! This continues the model's training on our recipe data
print("Fine-tuning started...")
trainer.train()
print("\nFine-tuning complete!")

#### Compare: Before vs After Fine-Tuning

The cell below generates text using the **fine-tuned** model, then loads a **fresh pre-trained** copy of GPT-2 and generates text with the same prompt. This lets you see the difference side by side.

**Change the prompt to whatever you like.**

In [ ]:
# Change the prompt below, then run this cell to compare before vs after.

prompt = "What is the temperature like today?"              # <-- change this
max_length = 100                                            # <-- (optional) change this

# Generate with the FINE-TUNED model (the one we just trained)
finetuned_output = generate_text(prompt, max_length, do_sample=True, top_k=50, temperature=0.7)

# Temporarily load a fresh PRE-TRAINED GPT-2 for comparison
import copy
finetuned_model = model  # save reference to fine-tuned model
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
pretrained_output = generate_text(prompt, max_length, do_sample=True, top_k=50, temperature=0.7)
model = finetuned_model  # restore the fine-tuned model

print("--- Pre-trained GPT-2 ---")
print(pretrained_output)
print()
print("--- Fine-tuned GPT-2 (recipes) ---")
print(finetuned_output)


> Key takeaway: After fine-tuning, the model is now much more likely to talk about recipes. This is best shown below, where we set the prompt to be something **completely unrelated to recipes**, and how the fine-tuned model just wants to talk about recipes. Keep in mind, though, in real practice, it is always important to write well-constructed and relevant prompts.

---
### Example 2: Fine-tune on full stop replacement

In this example, we fine-tune GPT-2 on ordinary sentences where **every full stop has been replaced with `*`**. For example:

> The cat sat on the mat* It looked up at the window* A bird flew past outside*

This is a purely formatting pattern with no connection to word choice or meaning. It is a good test of whether fine-tuning can teach a model a new surface-level habit.

**Run the cells below.**

In [ ]:
# Reload a fresh copy of GPT-2 so we start from scratch.
print("Reloading a fresh GPT-2 model...")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
print("Fresh model loaded!")

In [ ]:
# Ordinary sentences where every full stop has been replaced with *.
# Each example contains multiple sentences so the model can learn the pattern.
star_texts = [
    "The cat sat on the mat* It looked up at the window* A bird flew past outside*",
    "She walked to the shops* The sun was warm on her face* She bought a loaf of bread*",
    "He opened the door and stepped outside* The air was cold and crisp* He pulled his coat tighter*",
    "The children played in the garden* They chased each other around the trees* Their laughter filled the air*",
    "A large bird flew over the house* It circled twice before landing on the roof* Then it was gone*",
    "The rain began to fall* It tapped gently on the window* Soon the streets were wet*",
    "She picked up her bag and left* The door clicked shut behind her* The room fell silent*",
    "He sat down at his desk* There were papers scattered everywhere* He began sorting through them*",
    "The flowers bloomed in the garden* Their colours were vivid and bright* Bees moved slowly between them*",
    "A quiet voice called from the hall* Nobody answered at first* Then footsteps crossed the floor*",
    "The wind picked up in the afternoon* Leaves scattered across the path* The sky turned a deep grey*",
    "She filled the kettle and switched it on* While she waited she looked out at the garden* A robin sat on the fence*",
    "He read the letter twice* It was not the news he had hoped for* He folded it and put it away*",
    "The dog ran to the door as soon as it heard the key* It wagged its tail and barked* Everyone was home*",
    "The train arrived two minutes late* Passengers gathered their bags and stood* The doors slid open with a hiss*",
    "She set the table for dinner* There were four plates and four glasses* The smell of food filled the kitchen*",
    "He climbed the stairs slowly* His legs ached from the long walk* He was glad to be home*",
    "The library was quiet in the morning* Shelves of books lined every wall* A few people sat reading at tables*",
    "She poured a cup of tea and sat by the window* Outside the street was busy with people* She watched them pass*",
    "The meeting started late* Several people arrived after it had begun* Nobody seemed to mind*",
    "He planted seeds in the garden last spring* By summer they had grown into tall plants* The tomatoes were ready to pick*",
    "The road curved sharply to the left* Beyond it the valley opened up below* The view was remarkable*",
    "She finished the last page and closed the book* It had taken her three weeks to read it* She immediately wanted to start another*",
    "The shop was busy on Saturday morning* Baskets of fruit were stacked near the entrance* A queue had formed at the counter*",
    "He turned off the light and lay in the dark* The house was quiet* He could hear rain on the roof*",
    "The class started at nine each morning* Students arrived in ones and twos* The teacher waited until everyone was seated*",
    "She unpacked her suitcase slowly* Everything smelled faintly of sunscreen and sea air* The holiday felt far away already*",
    "He noticed the window had been left open* The curtains moved in the breeze* A few drops of rain had come in*",
    "The path through the woods was narrow* Roots crossed it at odd angles* She watched her footing carefully*",
    "He checked his watch and frowned* The bus was already ten minutes late* He decided to walk instead*",
    "The café was small and warm* Every table was taken* They waited near the door for a seat*",
    "She wrote the address on the envelope* Then she found a stamp in the drawer* She walked to the post box on the corner*",
    "The cat knocked the glass off the table* It shattered on the floor* She sighed and went to get the dustpan*",
    "He looked at the map for a long time* The route was not obvious* Eventually he asked someone for directions*",
    "The evening light was soft and golden* Shadows stretched across the lawn* Someone nearby was mowing their grass*",
    "She hung the washing on the line* The sheets billowed in the breeze* They would be dry by afternoon*",
    "He made a list of everything he needed* Then he lost the list somewhere in the kitchen* He wrote it again from memory*",
    "The park was quieter than usual* A few joggers passed along the path* Pigeons clustered near an empty bench*",
    "She answered the phone on the third ring* It was her sister calling from abroad* They talked for nearly an hour*",
    "He finished his coffee and rinsed the cup* The day had barely started* There was still a great deal to do*",
]

star_dataset = Dataset.from_dict({"text": star_texts})
star_tokenized = star_dataset.map(tokenize, batched=True, remove_columns=["text"])
print(f"Dataset prepared: {len(star_texts)} sentences")


In [ ]:
training_args = TrainingArguments(
    output_dir="./fine_tuned_stars",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    save_strategy="no",
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=star_tokenized,
    data_collator=data_collator,
)

print("Fine-tuning started...")
trainer.train()
print("\nFine-tuning complete!")


#### Compare: Before vs After Fine-Tuning

The cell below generates text using the **`*` fine-tuned** model, then compares it against a fresh pre-trained GPT-2. Try any ordinary sentence as your prompt and see whether the fine-tuned model uses `*` instead of full stops.


In [ ]:
# Change the prompt below, then run this cell to compare before vs after.

prompt = "What is the temperature like today?"              # <-- change this
max_length = 100                                            # <-- (optional) change this

# Generate with the FINE-TUNED model (the one we just trained)
finetuned_output = generate_text(prompt, max_length, do_sample=True, top_k=50, temperature=0.7)

# Temporarily load a fresh PRE-TRAINED GPT-2 for comparison
finetuned_model = model
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
pretrained_output = generate_text(prompt, max_length, do_sample=True, top_k=50, temperature=0.7)
model = finetuned_model  # restore the fine-tuned model

print("--- Pre-trained GPT-2 ---")
print(pretrained_output)
print()
print("--- Fine-tuned GPT-2 (stars) ---")
print(finetuned_output)


> Key takeaway: We attempted to fine-tune the model to predict '*' instead of a '.' . Because this is a simple, consistent substitution with no dependence on meaning or context, it is fairly easy for fine-tuning to pick up this pattern. 

---
### Example 3: Fine-tune on named entity counting

In this stage we fine-tune GPT-2 on sentences that end with a structured label counting the number of named entities (people's names) in that sentence. For example:

> Lucy is friends with Jack and they walk to school together. Output: Named_Entities=2

After fine-tuning, when you give the model a new sentence it should continue it with the correct `Output: Named_Entities=` label. This is a simple version of a real NLP task called **named entity recognition**.

**Run the cells below.**

In [ ]:
# Reload a fresh copy of GPT-2 so we start from scratch.
# Stage 2 changed the model's weights - we don't want those patterns
# interfering with our named entity fine-tuning.
print("Reloading a fresh GPT-2 model...")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
print("Fresh model loaded!")


In [ ]:
# Sentences labelled with the number of named entities (people's names) they contain.
# Each example ends with \n so the model learns to stop after the number.
ner_texts = [
    # 0 named entities
    "The cat sat on the mat. Output: Named_Entities=0\n",
    "A dog ran through the park and barked at the birds. Output: Named_Entities=0\n",
    "The teacher wrote on the board. Output: Named_Entities=0\n",
    "A bird flew over the house and landed in the garden. Output: Named_Entities=0\n",
    "The train arrived at the station on time. Output: Named_Entities=0\n",
    "A child played with a ball in the street. Output: Named_Entities=0\n",
    "The sun set slowly behind the clouds. Output: Named_Entities=0\n",
    "A book was left open on the table. Output: Named_Entities=0\n",
    # 1 named entity
    "Lucy went to the shops in the morning. Output: Named_Entities=1\n",
    "The teacher called on Jack to answer the question. Output: Named_Entities=1\n",
    "Tom arrived late to the meeting. Output: Named_Entities=1\n",
    "The dog followed Emma home from the park. Output: Named_Entities=1\n",
    "Oliver finished his homework before dinner. Output: Named_Entities=1\n",
    "Sarah bought a new coat from the market. Output: Named_Entities=1\n",
    "The letter was addressed to James. Output: Named_Entities=1\n",
    "Harry found a coin on the pavement. Output: Named_Entities=1\n",
    "Mia sang loudly in the shower. Output: Named_Entities=1\n",
    "The cat knocked over Grace's glass. Output: Named_Entities=1\n",
    # 2 named entities
    "Lucy is friends with Jack and they often walk to school together. Output: Named_Entities=2\n",
    "Tom and Sarah went for a walk along the river. Output: Named_Entities=2\n",
    "Emma told Jack that the film had already started. Output: Named_Entities=2\n",
    "Oliver and Mia sat next to each other on the bus. Output: Named_Entities=2\n",
    "The teacher asked Harry to help Grace with the task. Output: Named_Entities=2\n",
    "James called Tom to say he would be late. Output: Named_Entities=2\n",
    "Sarah lent her umbrella to Lucy. Output: Named_Entities=2\n",
    "The coach chose Oliver and Emma for the team. Output: Named_Entities=2\n",
    "Grace and Harry argued about which film to watch. Output: Named_Entities=2\n",
    "Mia waited for Tom outside the library. Output: Named_Entities=2\n",
    # 3 named entities
    "Alice, Bob, and Carol met for lunch at the café. Output: Named_Entities=3\n",
    "Emma told Jack that Oliver had called earlier. Output: Named_Entities=3\n",
    "Tom, Sarah, and Lucy planned a surprise party. Output: Named_Entities=3\n",
    "The teacher asked James, Harry, and Mia to stay behind. Output: Named_Entities=3\n",
    "Grace introduced Oliver to her friend Tom. Output: Named_Entities=3\n",
    "Sarah, Emma, and Jack decided to form a study group. Output: Named_Entities=3\n",
    "Alice passed a note to Bob while Carol watched. Output: Named_Entities=3\n",
    "Harry told Tom and Grace what he had seen. Output: Named_Entities=3\n",
    # 4 named entities
    "Alice, Bob, Carol, and David all arrived at the same time. Output: Named_Entities=4\n",
    "Tom introduced Sarah to Jack and Emma at the party. Output: Named_Entities=4\n",
    "Oliver, Mia, Harry, and Grace formed a band at school. Output: Named_Entities=4\n",
    "Lucy told James that both Tom and Sarah had already left. Output: Named_Entities=4\n",
]

ner_dataset = Dataset.from_dict({"text": ner_texts})
ner_tokenized = ner_dataset.map(tokenize, batched=True, remove_columns=["text"])
print(f"Dataset prepared: {len(ner_texts)} labelled sentences")


In [ ]:
training_args = TrainingArguments(
    output_dir="./fine_tuned_ner",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    save_strategy="no",
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ner_tokenized,
    data_collator=data_collator,
)

print("Fine-tuning started...")
trainer.train()
print("\nFine-tuning complete!")


#### Compare: Before vs After Fine-Tuning

The cell below generates text using the **named entity fine-tuned** model, then compares it against a fresh pre-trained GPT-2. The prompt ends with `Output: Named_Entities=` as we are just interested in the number that comes at the end.

In [ ]:
# Write a sentence containing names, followed by "Output: Named_Entities="
# Both models will then predict what token comes next - ideally the correct count.

prompt = "Tom and Sarah walked to the park. Output: Named_Entities="    # <-- change the sentence part

# We only want the model to generate a few tokens beyond the prompt (just the number).
# len(tokenizer(prompt)["input_ids"]) counts how many tokens the prompt itself uses,
# and we add 5 to allow a small amount of new output after it.
max_length = len(tokenizer(prompt)["input_ids"]) + 5

# The newline token (\n) was added to the end of every training example,
# so the model learned to generate it immediately after the number.
# Passing it as eos_token_id tells the generator to stop as soon as it produces \n.
newline_token_id = tokenizer.encode("\n")[0]

# Generate with the FINE-TUNED model
finetuned_output = generate_text(prompt, max_length, do_sample=False, eos_token_id=newline_token_id)

# Temporarily load a fresh PRE-TRAINED GPT-2 for comparison
finetuned_model = model
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
pretrained_output = generate_text(prompt, max_length, do_sample=False, eos_token_id=newline_token_id)
model = finetuned_model  # restore the fine-tuned model

print("--- Pre-trained GPT-2 ---")
print(pretrained_output)
print()
print("--- Fine-tuned GPT-2 (named entities) ---")
print(finetuned_output)


> Key takeaway: Both the pre-trained and fine-tuned models perform poorly at this task, often predicting the wrong number. This is expected, and it reveals an important limitation of fine-tuning a general language model on a task that requires **reasoning**. Correctly counting named entities is not just a pattern to memorise — it requires the model to identify which words are names, count them, and output that count. GPT-2 was not designed for this kind of structured reasoning; it learned to predict statistically likely next tokens, not to perform logical operations over a sentence.
>
> What fine-tuning *has* successfully done is teach the model the output format and the general concept that a number should follow the label. Getting the number right consistently would require a model architecture better suited to structured prediction or a much larger and more capable model.

---
### Your Own Example

Now it is your turn! Write your own training sentences in the cell below.

**Important tips:**
- Your sentences must share something in common - a **topic**, a **style**, or a **pattern** - so the model can learn from them
- For example: all about sports, all written like a pirate, all about your university, all in the style of a news headline
- **Come up with the sentences as a group** - aim for at least **20-30 sentences** together - the more you provide, the better the model will learn
- Each sentence should be a complete thought, roughly 1-2 lines long

---

#### Optional: experiment with training parameters

Once you have your data working, you can try adjusting the training settings below. **This is optional** — the defaults will work fine — but changing these values is a good way to build intuition for how fine-tuning works.

The key parameters you can change in `TrainingArguments` are:

- **`num_train_epochs`** — An *epoch* is one full pass through your entire training dataset. More epochs means the model sees each example more times, which can strengthen the learned pattern. Too many epochs on a small dataset can cause *overfitting*, where the model memorises the training sentences rather than generalising the pattern.

- **`learning_rate`** — Controls how much the model's weights are adjusted after each batch. A higher learning rate makes the model change faster but can make training unstable. A lower learning rate is more cautious but may need more epochs to see an effect.

- **`per_device_train_batch_size`** — How many training examples are processed together before the weights are updated. Smaller batches update more frequently but with noisier estimates; larger batches are more stable but use more memory.


In [ ]:
# We need to reload a fresh copy of GPT-2 so we start from scratch.
# Stage 3 changed the model's weights - we don't want those patterns
# interfering with our new fine-tuning.
print("Reloading a fresh GPT-2 model...")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
print("Fresh model loaded!")


In [ ]:
# WRITE YOUR OWN TRAINING SENTENCES BELOW
# As a group, come up with sentences and replace the examples with your own!
# Remember: all sentences should share a common theme or style.
# The model will learn from the patterns these sentences have in common.

my_data = [
    "Write your first sentence here.",
    "Write your second sentence here.",
    "Write your third sentence here.",
    "Write your fourth sentence here.",
    "Write your fifth sentence here.",
    # Keep adding more sentences below!
    # Aim for at least 20-30 sentences.
]

print(f"You have written {len(my_data)} sentences.")
if len(my_data) < 20:
    print("Try to add more sentences for better results! Aim for at least 20.")
else:
    print("Good amount of data!")

In [ ]:
# Prepare your data the same way we did in Stage 1:
# convert to a Dataset, then tokenise it
custom_dataset = Dataset.from_dict({"text": my_data})
custom_tokenized = custom_dataset.map(tokenize, batched=True, remove_columns=["text"])

# Same training settings as Stage 1
training_args = TrainingArguments(
    output_dir="./fine_tuned_custom",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    save_strategy="no",
    logging_steps=5,
    report_to="none",
)

# Create the trainer and start fine-tuning on your data
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=custom_tokenized,
    data_collator=data_collator,
)

print("Fine-tuning on your data...")
trainer.train()
print("\nFine-tuning complete!")

In [ ]:
# Test your custom fine-tuned model!
# Change the prompt to something related to your group's theme.
# For example, if your sentences were about sports, try a sports-related prompt.
# Run this cell multiple times with different prompts to see how well
# the model picked up on your group's data.

prompt = "Prompt here"                             # <-- change this
max_length = 100                                   # <-- change this

output = generate_text(prompt, max_length, do_sample=True, top_k=50, temperature=0.7)
print("--- Your Fine-tuned GPT-2 ---")
print(output)

---
## All Done!

Once you have completely finished with this notebook, **disconnect your runtime** to free up the GPU for others:

**Runtime > Disconnect and delete runtime**

Then head back to the tutorial website to complete any remaining discussion questions with your group.
